# Check the clisops subset operator against CMIP6 QA/QC findings

Version 2026-11-12

The final overview covers all 14 CMIP6 QA/QC findings. The synthetic tests focus on the findings selected as relevant to the clisops subset operator. They check xarray decoding and the expected combined time and geographic bounding-box subset.

| Finding | Metadata issue |
| --- | --- |
| [3.1] | Unpadded or hyphen-separated reference time in the time-units string |
| [3.3] | Missing coordinate `axis` or `standard_name` |
| [3.7] | Projected x/y axes with degree units |
| [3.8] | Area variable without a geographic coordinate reference |
| [3.12] | Reference to an absent auxiliary `height` coordinate |

The candidate examples below give model, experiment, and variant only. They are partial ESGF dataset identifiers for future real-data validation. This notebook tests only the synthetic fixtures.

In [1]:
import sys
from importlib.metadata import version

import ipytest
import numpy as np
import pytest
import xarray as xr
from IPython.display import Markdown, display
from clisops.ops.subset import subset
from netCDF4 import Dataset

ipytest.autoconfig(raise_on_error=True, magics=False, addopts=())

print(f"Python {sys.version.split()[0]}")
for package in ("clisops", "xarray", "cf-xarray", "numpy", "netCDF4", "cftime", "pytest", "ipytest"):
    print(f"{package} {version(package)}")

Python 3.13.18
clisops 0.18.1
xarray 2025.10.1
cf-xarray 0.11.3
numpy 2.5.4
netCDF4 1.7.5
cftime 1.6.5
pytest 9.2.0
ipytest 0.14.2


## Shared test data

Use four daily samples from 1 to 4 January 1850 and request the middle two days. The bbox is 12°W to 3°W and 43°N to 52°N. On the rectilinear grid it includes latitudes 45° and 50° and longitudes 10°W and 5°W. Unique data values make a shifted selection visible.

In [2]:
TIME_UNITS = "days since 1850-01-01 00:00:00"
DATES = np.arange("1850-01-01", "1850-01-05", dtype="datetime64[D]").astype("datetime64[ns]")
TIME_RANGE = "1850-01-02T00:00:00/1850-01-03T00:00:00"
BBOX = (-12, 43, -3, 52)  # west, south, east, north

time = (
    "time",
    np.arange(4),
    {"units": TIME_UNITS, "calendar": "standard", "axis": "T", "standard_name": "time"},
)
latitude = (
    "lat",
    [40, 45, 50, 55],
    {"units": "degrees_north", "axis": "Y", "standard_name": "latitude"},
)
longitude = (
    "lon",
    [-15, -10, -5, 0, 5],
    {"units": "degrees_east", "axis": "X", "standard_name": "longitude"},
)
pressure = (
    "plev",
    [100000, 85000],
    {"units": "Pa", "axis": "Z", "standard_name": "air_pressure", "positive": "down"},
)

rectilinear = xr.Dataset(
    {"ta": (("time", "plev", "lat", "lon"), np.arange(160.0).reshape(4, 2, 4, 5))},
    coords={"time": time, "plev": pressure, "lat": latitude, "lon": longitude},
)
rectilinear_bbox = {"lat": slice(1, 3), "lon": slice(1, 3)}

In [3]:
FINDINGS = {
    "3.1": "Time-unit string formatting",
    "3.2": "Temporal coordinate positioning",
    "3.3": "Coordinate axis / standard_name metadata",
    "3.4": "Parent-simulation controlled vocabulary",
    "3.5": "Physical-variable standard_name mismatches",
    "3.6": "Equivalent numeric notation for sos units",
    "3.7": "Horizontal x/y coordinate units",
    "3.8": "Geographic referencing of area",
    "3.9": "Recommended metadata on bnds",
    "3.10": "Ensemble-member index inconsistencies",
    "3.11": "Global realization_index attribute type",
    "3.12": "Missing auxiliary height coordinate",
    "3.13": "Time and spatial bounds inconsistencies",
    "3.14": "Nominal-resolution vocabulary",
}

cases = []


def add_case(finding, name, raw, variable, bbox_index, bbox_mask=None):
    # Use positional expectations, independently of clisops' coordinate discovery.
    expected = raw[variable].assign_coords(time=DATES)
    expected_subset = expected.isel(time=slice(1, 3), **bbox_index)
    if bbox_mask is not None:
        expected_subset = expected_subset.where(bbox_mask)
    cases.append(pytest.param((raw, expected, expected_subset), id=f"{finding}-{name}"))

## Finding [3.1] — Time-unit string formatting

*Representative diagnostic*  
The reference time in the units string uses unpadded fields or hyphens as separators.

**Exact metadata reproduction.** The variants write `days since 1850-01-01 0:0:0.0` and `days since 1850-01-01-00-00-00` to NetCDF. The control uses `00:00:00` with the same offsets and standard calendar. Each variant must decode to the expected dates and return the expected time and bbox subset. A decoding exception appears as a pytest setup error.

Candidate for future real-data validation — `CanESM5` / `historical` / `r1i1p1f1`.

In [4]:
add_case("3.1+3.3", "rectilinear-control", rectilinear, "ta", rectilinear_bbox)

short_time_units = rectilinear.copy(deep=True)
short_time_units.time.attrs["units"] = "days since 1850-01-01 0:0:0.0"
add_case("3.1", "unpadded-time-units", short_time_units, "ta", rectilinear_bbox)

hyphenated_time_units = rectilinear.copy(deep=True)
hyphenated_time_units.time.attrs["units"] = "days since 1850-01-01-00-00-00"
add_case("3.1", "hyphenated-time-units", hyphenated_time_units, "ta", rectilinear_bbox)

## Finding [3.2] — Temporal coordinate positioning

Only some temporal shifts are currently classified as non-blocking. Those shifts were attributed to artefacts in the QA/QC implementation. This finding is included for context.

## Finding [3.3] — Coordinate axis / standard_name metadata

*Representative diagnostic*  
A coordinate is missing its `axis` or `standard_name` attribute.

**Reported pattern and robustness probes.** The eight variants remove one attribute at a time from `time`, `lat`, `lon`, or `plev`. Each keeps the coordinate name, units, and other attributes. These isolated removals check whether the expected time and bbox selection still works and preserves both pressure levels.

Candidate for future real-data validation — `CNRM-CM6-1` / `historical` / `r1i1p1f2`.

In [5]:
for coordinate in ("time", "lat", "lon", "plev"):
    for attribute in ("axis", "standard_name"):
        missing_attribute = rectilinear.copy(deep=True)
        del missing_attribute[coordinate].attrs[attribute]
        add_case("3.3", f"{coordinate}-without-{attribute}", missing_attribute, "ta", rectilinear_bbox)

## Finding [3.7] — Horizontal x/y coordinate units

*Representative diagnostic*  
Projected horizontal coordinates declare `degrees` where metre units are expected.

**Reported metadata pattern.** The variant gives `projection_x_coordinate` and `projection_y_coordinate` axes degree values and units. The metre-valued control and the variant share valid 2-D latitude and longitude. The combined test expects a 2 × 2 window with the northeast data cell masked at both selected times and the geographic coordinates preserved. The separate negative control is a robustness probe that removes both geographic coordinates.

Candidate for future real-data validation — `MIROC6` / `historical` / `r1i1p1f1`.

In [6]:
curvilinear = xr.Dataset(
    {"tas": (("time", "y", "x"), np.arange(48.0).reshape(4, 3, 4))},
    coords={
        "time": time,
        "x": ("x", [0, 500_000, 1_000_000, 1_500_000],
              {"standard_name": "projection_x_coordinate", "units": "m", "axis": "X"}),
        "y": ("y", [0, 500_000, 1_000_000],
              {"standard_name": "projection_y_coordinate", "units": "m", "axis": "Y"}),
        "lat": (("y", "x"), [[39, 40, 41, 42], [43, 45, 46, 47], [49, 50, 53, 54]],
                {"standard_name": "latitude", "units": "degrees_north"}),
        "lon": (("y", "x"), [[-16, -11, -6, -1], [-15, -10, -5, 0], [-14, -9, -4, 1]],
                {"standard_name": "longitude", "units": "degrees_east"}),
    },
)
curvilinear_bbox = {"y": slice(1, 3), "x": slice(1, 3)}
curvilinear_mask = xr.DataArray([[True, True], [True, False]], dims=("y", "x"))

add_case("3.7+3.8", "projected-axes-in-metres", curvilinear, "tas", curvilinear_bbox, curvilinear_mask)

axes_in_degrees = curvilinear.assign_coords(
    x=("x", [0, 5, 10, 15], {**curvilinear.x.attrs, "units": "degrees"}),
    y=("y", [0, 5, 10], {**curvilinear.y.attrs, "units": "degrees"}),
)
add_case("3.7", "projected-axes-in-degrees", axes_in_degrees, "tas", curvilinear_bbox, curvilinear_mask)

## Finding [3.8] — Geographic referencing of area

*Representative diagnostic*  
The auxiliary `area(y, x)` variable lacks a geographic coordinate reference.

**Reported metadata pattern.** The NetCDF fixture omits the `coordinates` attribute from `area` and keeps the latitude and longitude reference on `tas`. The combined request checks the expected `tas` subset and the cropped, masked `area` values with their geographic coordinates.

In [7]:
area_without_reference = curvilinear.assign(
    area=(("y", "x"), 1e6 + np.arange(12.0).reshape(3, 4), {"units": "m2"}),
)
area_without_reference.area.encoding["coordinates"] = None
expected_area = area_without_reference.area.isel(curvilinear_bbox).where(curvilinear_mask)
add_case("3.8", "area-without-coordinate-reference",
         area_without_reference, "tas", curvilinear_bbox, curvilinear_mask)

## Finding [3.12] — Missing auxiliary height coordinate

*Representative diagnostic*  
The data variable references `height`, which is absent.

**Exact reference reproduction.** Both files declare `coordinates="height"` on `tas`. The control has a scalar 2 m height and the variant omits it. The fixture checks that the reference survives on disk and in `tas.encoding["coordinates"]` after xarray decoding. The combined test checks whether the dangling reference affects the returned data or coordinates.

Candidate for future real-data validation — `KIOST-ESM` / `historical` / `r1i1p1f1`.

In [8]:
height_control = xr.Dataset(
    {"tas": (("time", "lat", "lon"), np.arange(80.0).reshape(4, 4, 5), {"coordinates": "height"})},
    coords={
        "time": time,
        "lat": latitude,
        "lon": longitude,
        "height": ((), 2.0, {"standard_name": "height", "units": "m", "positive": "up"}),
    },
)
add_case("3.12", "height-control", height_control, "tas", rectilinear_bbox)
missing_height = height_control.drop_vars("height")
add_case("3.12", "missing-height", missing_height, "tas", rectilinear_bbox)

## Pytest tests

The fixture verifies NetCDF metadata and xarray decoding before running the combined request. The negative control checks the error for a bbox request on a projected grid without geographic coordinates.

In [9]:
ipytest.clean()


@pytest.fixture(params=cases)
def decoded_case(request, tmp_path):
    raw, expected, expected_subset = request.param
    path = tmp_path / "fixture.nc"
    raw.to_netcdf(path, engine="netcdf4")

    with Dataset(path) as file:
        assert file["time"].units == raw["time"].attrs["units"]
        if raw is area_without_reference:
            assert {"lat", "lon"} <= set(file["tas"].coordinates.split())
            assert "coordinates" not in file["area"].ncattrs()
        if raw is height_control or raw is missing_height:
            assert file[expected.name].coordinates == "height"
            assert ("height" in file.variables) == (raw is height_control)

    with xr.open_dataset(path, engine="netcdf4") as dataset:
        decoded = dataset.load()

    assert decoded[expected.name].equals(expected)
    for name, coordinate in raw.coords.items():
        for attr in ("axis", "standard_name", "units"):
            actual = decoded[name].encoding.get(attr, decoded[name].attrs.get(attr))
            assert actual == coordinate.attrs.get(attr), f"{name}.{attr}: {actual!r}"
    if raw is area_without_reference:
        assert decoded.area.encoding.get("coordinates") is None
    if raw is missing_height:
        assert decoded[expected.name].encoding.get("coordinates") == "height"
        assert "height" not in decoded.variables

    return decoded, expected_subset


def test_time_and_bbox_subset(decoded_case):
    decoded, expected_subset = decoded_case
    outputs = subset(decoded, time=TIME_RANGE, area=BBOX, output_type="xarray")
    assert len(outputs) == 1
    assert outputs[0][expected_subset.name].equals(expected_subset)
    if "area" in decoded.data_vars:
        assert outputs[0].area.equals(expected_area)


@pytest.mark.parametrize("source", [pytest.param(axes_in_degrees, id="3.7-missing-geographic-coordinates")])
def test_bbox_without_geographic_coordinates(source):
    without_lat_lon = xr.decode_cf(source.drop_vars(["lat", "lon"]))
    with pytest.raises(AttributeError, match="has no attribute 'lon'") as error:
        subset(without_lat_lon, area=BBOX, output_type="xarray")

    print(f"Bbox without geographic coordinates raised {error.type.__name__}: {error.value}")

## Run pytest

In [10]:
class FindingResults:
    def __init__(self):
        self.tests = {finding: [] for finding in FINDINGS}
        self.outcomes = {}

    def pytest_itemcollected(self, item):
        # A shared control can contribute to more than one finding.
        for finding in item.callspec.id.split("-", 1)[0].split("+"):
            self.tests[finding].append(item.nodeid)

    def pytest_runtest_logreport(self, report):
        phase_outcomes = self.outcomes.setdefault(report.nodeid, {})
        phase_outcomes[report.when] = "skipped" if hasattr(report, "wasxfail") else report.outcome

    def status(self, finding):
        outcomes = [self.outcomes.get(nodeid, {}) for nodeid in self.tests[finding]]
        passed = {"setup": "passed", "call": "passed", "teardown": "passed"}
        if any("failed" in phases.values() for phases in outcomes):
            return "FAIL"
        if outcomes and all(phases == passed for phases in outcomes):
            return "PASS"
        return "INCOMPLETE"


results = FindingResults()
pytest_exit = ipytest.run(
    "-v", "-rP", "--color=no", "-p", "no:cacheprovider",
    plugins=[results], raise_on_error=False,
)

======================================= test session starts ========================================
platform darwin -- Python 3.13.18, pytest-9.2.0, pluggy-1.7.0 -- /private/tmp/c3s-cmip6-update-tests-01a1cbf2/bin/python
rootdir: /Users/pingu/Documents/GitHub/macpingu/c3s-cmip6-update-tests/notebooks
plugins: platformdirs-4.14.5, anyio-4.16.0
collecting ... collected 17 items

t_97d8f9f2c6b3408ea3e03950da63d9b7.py::test_time_and_bbox_subset[3.1+3.3-rectilinear-control] PASSED [  5%]
t_97d8f9f2c6b3408ea3e03950da63d9b7.py::test_time_and_bbox_subset[3.1-unpadded-time-units] PASSED [ 11%]
t_97d8f9f2c6b3408ea3e03950da63d9b7.py::test_time_and_bbox_subset[3.1-hyphenated-time-units] PASSED [ 17%]
t_97d8f9f2c6b3408ea3e03950da63d9b7.py::test_time_and_bbox_subset[3.3-time-without-axis] PASSED [ 23%]
t_97d8f9f2c6b3408ea3e03950da63d9b7.py::test_time_and_bbox_subset[3.3-time-without-standard_name] PASSED [ 29%]
t_97d8f9f2c6b3408ea3e03950da63d9b7.py::test_time_and_bbox_subset[3.3-lat-without-axis] P

## Results and limitations

Each finding includes its controls. Findings [3.1] and [3.3] share the rectilinear control. Findings [3.7] and [3.8] share the metre-valued curvilinear control. For [3.7], the negative control passes when the request raises the expected missing-longitude error. A PASS requires successful setup, execution, and teardown for every relevant test. For untested findings, low expected relevance is a judgement about the combined time and bbox request. The Woodpecker column distinguishes that judgement from the observed synthetic results.

The fixtures omit longitude cell bounds, so clisops can warn about them during bbox selection. The tests cover isolated metadata changes in small, single-file fixtures with one calendar and a fixed time range and bbox. Compatibility across the affected real datasets remains unverified.

In [11]:
low_relevance = "Low expected relevance"
untested_assessments = {
    "3.2": "Not evaluated. Only some temporal shifts are classified as non-blocking.",
    "3.4": low_relevance,
    "3.5": "Not evaluated",
    "3.6": low_relevance,
    "3.9": low_relevance,
    "3.10": low_relevance,
    "3.11": low_relevance,
    "3.13": "Excluded from the current assessment (already classified HIGH)",
    "3.14": low_relevance,
}

table = [
    "| Finding | Description | Assessment | Subsetting result | Woodpecker required? |",
    "| --- | --- | --- | --- | --- |",
]
for finding, description in FINDINGS.items():
    assessment = untested_assessments.get(finding)
    if assessment is None:
        assessment = "Evaluated with synthetic fixtures"
        status = results.status(finding)
        woodpecker = "Not needed (synthetic)" if status == "PASS" else "To investigate"
    else:
        status = "Not run"
        woodpecker = "Unlikely needed" if assessment == low_relevance else "Undetermined"
    table.append(f"| [{finding}] | {description} | {assessment} | {status} | {woodpecker} |")

display(Markdown("\n".join(table)))
assert pytest_exit == pytest.ExitCode.OK, f"pytest exited with {pytest_exit.name}"

| Finding | Description | Assessment | Subsetting result | Woodpecker required? |
| --- | --- | --- | --- | --- |
| [3.1] | Time-unit string formatting | Evaluated with synthetic fixtures | PASS | Not needed (synthetic) |
| [3.2] | Temporal coordinate positioning | Not evaluated. Only some temporal shifts are classified as non-blocking. | Not run | Undetermined |
| [3.3] | Coordinate axis / standard_name metadata | Evaluated with synthetic fixtures | PASS | Not needed (synthetic) |
| [3.4] | Parent-simulation controlled vocabulary | Low expected relevance | Not run | Unlikely needed |
| [3.5] | Physical-variable standard_name mismatches | Not evaluated | Not run | Undetermined |
| [3.6] | Equivalent numeric notation for sos units | Low expected relevance | Not run | Unlikely needed |
| [3.7] | Horizontal x/y coordinate units | Evaluated with synthetic fixtures | PASS | Not needed (synthetic) |
| [3.8] | Geographic referencing of area | Evaluated with synthetic fixtures | PASS | Not needed (synthetic) |
| [3.9] | Recommended metadata on bnds | Low expected relevance | Not run | Unlikely needed |
| [3.10] | Ensemble-member index inconsistencies | Low expected relevance | Not run | Unlikely needed |
| [3.11] | Global realization_index attribute type | Low expected relevance | Not run | Unlikely needed |
| [3.12] | Missing auxiliary height coordinate | Evaluated with synthetic fixtures | PASS | Not needed (synthetic) |
| [3.13] | Time and spatial bounds inconsistencies | Excluded from the current assessment (already classified HIGH) | Not run | Undetermined |
| [3.14] | Nominal-resolution vocabulary | Low expected relevance | Not run | Unlikely needed |